In [1]:
# Install required packages
!pip install torch>=2.0.0 transformers>=4.36.0 datasets>=2.14.0 accelerate>=0.24.0
!pip install bitsandbytes>=0.41.0 flash-attn>=2.3.0 huggingface-hub>=0.19.0
!pip install peft>=0.7.0 trl>=0.7.0 numpy>=1.24.0 scipy>=1.10.0
!pip install scikit-learn>=1.3.0 tqdm>=4.65.0 sentencepiece>=0.1.99 protobuf>=3.20.0


In [2]:
# Import required libraries
import os
import json
import torch
import logging
from typing import Dict, List, Any
from dataclasses import dataclass
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import numpy as np
from huggingface_hub import login

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA available: True
GPU: NVIDIA L4
GPU Memory: 23.80 GB


In [4]:
@dataclass
class TrainingConfig:
    """Configuration for model training"""
    # Model settings - MISTRAL
    base_model: str = "mistralai/Mistral-7B-Instruct-v0.2"
    model_name: str = "bi-intent-discovery-mistral"

    # Training settings - OPTIMIZED FOR A100
    num_epochs: int = 5
    batch_size: int = 3  # Conservative for stability
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-5
    warmup_steps: int = 200
    max_seq_length: int = 1024  # Conservative

    # Data settings
    train_data_path: str = "training_data_500_examples.json"
    prompt_template_path: str = "prompt_new.txt"

    # Output settings
    output_dir: str = "./trained_model"
    save_to_hf: bool = True
    hf_username: str = "ssuki"

    # Hardware settings
    use_4bit: bool = False
    use_8bit: bool = False
    use_flash_attention: bool = False

# Create config instance
config = TrainingConfig()

print("Training Configuration:")
print(f"Base Model: {config.base_model}")
print(f"Training Epochs: {config.num_epochs}")
print(f"Batch Size: {config.batch_size}")
print(f"Gradient Accumulation: {config.gradient_accumulation_steps}")
print(f"Effective Batch Size: {config.batch_size * config.gradient_accumulation_steps}")
print(f"Learning Rate: {config.learning_rate}")
print(f"Warmup Steps: {config.warmup_steps}")
print(f"Max Sequence Length: {config.max_seq_length}")
print(f"Output Directory: {config.output_dir}")
print(f"Save to HF: {config.save_to_hf}")
if config.hf_username:
    print(f"HF Username: {config.hf_username}")

Training Configuration:
Base Model: Qwen/Qwen2.5-7B-Instruct
Training Epochs: 5
Batch Size: 2
Gradient Accumulation: 6
Effective Batch Size: 12
Learning Rate: 1e-05
Warmup Steps: 200
Max Sequence Length: 1024
Output Directory: ./trained_model
Save to HF: True
HF Username: ssuki


In [5]:
from google.colab import files

print("Upload your training data file (training_data_500_examples.json):")
uploaded = files.upload()

print("\nUpload your prompt template file (prompt_new.txt):")
uploaded_prompt = files.upload()

# Move files to correct locations
if 'training_data_500_examples.json' in uploaded:
    with open('training_data_500_examples.json', 'wb') as f:
        f.write(uploaded['training_data_500_examples.json'])
    print("Training data uploaded successfully")

if 'prompt_new.txt' in uploaded_prompt:
    with open('prompt_new.txt', 'wb') as f:
        f.write(uploaded_prompt['prompt_new.txt'])
    print("Prompt template uploaded successfully")

Upload your training data file (training_data_500_examples.json):


Saving training_data_500_examples.json to training_data_500_examples.json

Upload your prompt template file (prompt_new.txt):


Saving prompt_new.txt to prompt_new.txt
Training data uploaded successfully
Prompt template uploaded successfully


In [6]:
class BIIntentTrainer:
    """Trainer for BI Intent Discovery model"""

    def __init__(self, config: TrainingConfig):
        self.config = config
        self.tokenizer = None
        self.model = None
        self.trainer = None

    def load_prompt_template(self) -> str:
        """Load the prompt template"""
        try:
            with open(self.config.prompt_template_path, 'r', encoding='utf-8') as f:
                return f.read().strip()
        except FileNotFoundError:
            logger.warning(f"Prompt template not found at {self.config.prompt_template_path}")
            return self._get_default_prompt()

    def _get_default_prompt(self) -> str:
        """Default prompt template if file not found"""
        return """# BI Planning & Discovery Agent
You are an AI assistant specialized in analyzing natural language BI questions and breaking them into structured steps for query building.

## Phases
### Phase 1: Planning
- Detect if question is **complex** (multi-step, dependencies, ranking, comparison, or time-based logic).
- Complexity indicators: "for the X", "top/best/highest/lowest X", "X that are Y", "based on X", "compare X with Y", "X for those Y".
- If complex:
  1. Extract BI elements (measures, dimensions, time, filters).
  2. Break into ordered steps (like CTEs).
  3. Add post-processing (ranking, sorting, formatting).
- If simple: skip planning.

### Phase 2: Discovery
For each question or planning step:
1. Extract BI concepts (measures, dimensions, timeframes, timegrain, patterns, filters, segments, breakdowns).
2. Map exact phrases to BI fields (store in `original_phrase`).
3. Capture **all unmatched terms** in `unmatched_intents` with `phrase`, `type`, and `reason`.
4. Handle **ambiguity**: If a phrase can mean multiple things, request clarification.

## Output Format
Respond with a JSON object containing your intent and discovery results.

## Question: {question}

## Response:"""

    def load_training_data(self) -> List[Dict[str, Any]]:
        """Load and preprocess training data"""
        logger.info(f"Loading training data from {self.config.train_data_path}")

        try:
            with open(self.config.train_data_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except FileNotFoundError:
            raise FileNotFoundError(f"Training data not found at {self.config.train_data_path}")

        logger.info(f"Loaded {len(data)} training examples")
        return data

    def format_training_example(self, example: Dict[str, Any], prompt_template: str) -> str:
        """Format a training example into the model's expected format"""
        question = example["input"]
        expected_output = json.dumps(example["output"], ensure_ascii=False, indent=2)

        # Format the prompt
        formatted_prompt = prompt_template.format(question=question)

        # Create the full training text
        training_text = f"{formatted_prompt}\n{expected_output}"

        return training_text

    def prepare_dataset(self, data: List[Dict[str, Any]], prompt_template: str) -> Dataset:
        """Prepare the dataset for training"""
        logger.info("Preparing dataset...")

        formatted_examples = []
        for example in data:
            formatted_text = self.format_training_example(example, prompt_template)
            formatted_examples.append({"text": formatted_text})

        logger.info(f"Successfully formatted {len(formatted_examples)} examples")

        # Create dataset
        dataset = Dataset.from_list(formatted_examples)
        return dataset

    def load_model_and_tokenizer(self):
        """Load the base model and tokenizer"""
        logger.info(f"Loading model: {self.config.base_model}")

        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config.base_model,
            trust_remote_code=True,
            padding_side="right"
        )

        # Add padding token if not present
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Load model with optimizations
        model_kwargs = {
            "trust_remote_code": True,
            "torch_dtype": torch.float16,
        }

        if self.config.use_4bit:
            from transformers import BitsAndBytesConfig
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4"
            )
            model_kwargs["quantization_config"] = quantization_config
        elif self.config.use_8bit:
            model_kwargs["load_in_8bit"] = True

        if self.config.use_flash_attention:
            model_kwargs["attn_implementation"] = "flash_attention_2"

        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model,
            **model_kwargs
        )

        # Enable gradient checkpointing for memory efficiency
        self.model.gradient_checkpointing_enable()

        logger.info("Model and tokenizer loaded successfully")

    def tokenize_function(self, examples):
        """Tokenize the dataset"""
        return self.tokenizer(
            examples["text"],
            truncation=True,
            padding=True,
            max_length=self.config.max_seq_length,
            return_tensors="pt"
        )

    def setup_training(self, dataset: Dataset):
        """Setup the training configuration"""
        logger.info("Setting up training...")

        # Tokenize dataset
        tokenized_dataset = dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=dataset.column_names
        )

        # Data collator
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False
        )

        # Training arguments - MEMORY OPTIMIZED
        training_args = TrainingArguments(
            output_dir=self.config.output_dir,
            num_train_epochs=self.config.num_epochs,
            per_device_train_batch_size=self.config.batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            learning_rate=self.config.learning_rate,
            warmup_steps=self.config.warmup_steps,
            logging_steps=10,
            save_steps=200,
            eval_steps=200,
            fp16=False,
            dataloader_pin_memory=False,
            remove_unused_columns=False,
            report_to=None,
            # MEMORY OPTIMIZATIONS
            weight_decay=0.01,
            max_grad_norm=1.0,
            lr_scheduler_type="cosine",
            dataloader_num_workers=0,  # REDUCED for memory
            group_by_length=False,  # DISABLED to save memory
            # ADDITIONAL MEMORY SAVINGS
            gradient_checkpointing=True,  # ENABLED
            optim="adamw_torch",  # Use PyTorch optimizer
            dataloader_drop_last=True,  # Drop incomplete batches
        )

        # Initialize trainer
        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_dataset,
            eval_dataset=tokenized_dataset.select(range(min(50, len(tokenized_dataset)))),  # REDUCED for memory
            data_collator=data_collator,
            tokenizer=self.tokenizer,
        )

        logger.info("Training setup completed")

    def train(self):
        """Execute the training process"""
        logger.info("Starting training...")

        # Load prompt template
        prompt_template = self.load_prompt_template()

        # Load and prepare data
        raw_data = self.load_training_data()
        dataset = self.prepare_dataset(raw_data, prompt_template)

        # Load model and tokenizer
        self.load_model_and_tokenizer()

        # Setup training
        self.setup_training(dataset)

        # Start training
        logger.info("Training started...")
        train_result = self.trainer.train()

        # Save the model
        logger.info("Saving model...")
        self.trainer.save_model()
        self.tokenizer.save_pretrained(self.config.output_dir)

        # Save training metrics
        metrics = train_result.metrics
        with open(os.path.join(self.config.output_dir, "training_metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

        logger.info(f"Training completed. Metrics: {metrics}")

        return train_result

    def save_to_huggingface(self):
        """Save the trained model to Hugging Face Hub"""
        if not self.config.save_to_hf:
            logger.info("Skipping Hugging Face upload (save_to_hf=False)")
            return

        if not self.config.hf_username:
            logger.warning("HF username not provided, skipping upload")
            return

        try:
            # Login to Hugging Face
            login()

            # Model name for HF
            model_name = f"{self.config.hf_username}/{self.config.model_name}"

            logger.info(f"Uploading model to Hugging Face: {model_name}")

            # Push model and tokenizer
            self.model.push_to_hub(model_name)
            self.tokenizer.push_to_hub(model_name)

            # Create model card
            self._create_model_card(model_name)

            logger.info(f"Model successfully uploaded to: https://huggingface.co/{model_name}")

        except Exception as e:
            logger.error(f"Error uploading to Hugging Face: {e}")

    def _create_model_card(self, model_name: str):
        """Create a model card for the uploaded model"""
        model_card = f"""---
language:
- en
tags:
- bi-intent-discovery
- business-intelligence
- question-analysis
- structured-output
license: mit
---

# BI Intent Discovery Model

This model is fine-tuned from Qwen2.5-7B-Instruct to perform Business Intelligence (BI) intent discovery tasks.

## Model Description

The model analyzes natural language questions about business intelligence data and breaks them down into structured steps for query building. It performs two main phases:

1. **Planning Phase**: Detects complex questions and breaks them into ordered steps
2. **Discovery Phase**: Extracts BI concepts (measures, dimensions, timeframes, etc.) from questions

## Training Data

- 500 examples of BI questions with structured outputs
- Covers various complexity levels from simple to multi-step queries
- Includes examples with ambiguity handling and unmatched intent capture

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

# Load model
model_name = "{model_name}"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

# Example usage
question = "Show me total sales by region for the last quarter"
# Format with your prompt template and generate response
```

## Output Format

The model outputs structured JSON containing:
- Intent classification
- Discovery results with measures, dimensions, timeframes
- Unmatched intents for ambiguous terms
- Step-by-step breakdown for complex queries

## Training Configuration

- Base Model: Qwen2.5-7B-Instruct
- Training Examples: 500
- Epochs: {self.config.num_epochs}
- Learning Rate: {self.config.learning_rate}
- Max Sequence Length: {self.config.max_seq_length}
"""

        # Save model card
        card_path = os.path.join(self.config.output_dir, "README.md")
        with open(card_path, "w", encoding="utf-8") as f:
            f.write(model_card)

        # Upload model card
        try:
            from huggingface_hub import upload_file
            upload_file(
                path_or_fileobj=card_path,
                path_in_repo="README.md",
                repo_id=model_name,
                repo_type="model"
            )
        except Exception as e:
            logger.warning(f"Could not upload model card: {e}")

In [7]:
# Clear GPU memory
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

# Initialize trainer
trainer = BIIntentTrainer(config)

# Start training
print("Starting training process...")
train_result = trainer.train()

print("Training completed!")
print(f"Final training loss: {train_result.training_loss:.4f}")

Starting training process...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sukruthisantosh (sukruthisantosh-imperial-college-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


OutOfMemoryError: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 22.16 GiB of which 59.38 MiB is free. Process 7374 has 22.10 GiB memory in use. Of the allocated memory 21.60 GiB is allocated by PyTorch, and 274.04 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Save to Hugging Face
if config.save_to_hf and config.hf_username:
    print("Uploading model to Hugging Face...")
    trainer.save_to_huggingface()
    print("Upload completed!")
else:
    print("Skipping Hugging Face upload")

In [ ]:
# Create a zip file of the trained model
import zipfile

def zip_model_files():
    """Create a zip file of the trained model"""
    zip_path = "trained_model.zip"

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(config.output_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, config.output_dir)
                zipf.write(file_path, arcname)

    return zip_path

# Create zip file
zip_path = zip_model_files()
print(f"Model files zipped to: {zip_path}")

# Download the zip file
from google.colab import files
files.download(zip_path)

In [ ]:
import time
import json
from datetime import datetime

def test_model(question: str):
    """Test the trained model with a sample question"""
    # Load the trained model
    model_path = config.output_dir

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    # Load prompt template
    prompt_template = trainer.load_prompt_template()
    formatted_prompt = prompt_template.format(question=question)

    # Generate response with timing
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    # Start timing
    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # End timing
    end_time = time.time()
    inference_time = end_time - start_time

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the response part (after the prompt)
    response_text = response[len(formatted_prompt):].strip()

    return response_text, inference_time

# Test multiple questions and save results
test_questions = [
    "Show me total sales by region for the last quarter",
    "What is the daily average number of customers who complete their subscription renewals?",
    "Compare current year revenue with last year's performance",
    "List the top 5 performing products by revenue"
]

print("Testing multiple questions for inference time analysis...")
print("=" * 60)

results = {
    "test_timestamp": datetime.now().isoformat(),
    "model_name": config.model_name,
    "base_model": config.base_model,
    "test_questions": [],
    "summary": {}
}

total_time = 0
times = []

for i, question in enumerate(test_questions, 1):
    print(f"\nTest {i}: {question}")
    try:
        response, inference_time = test_model(question)
        times.append(inference_time)
        total_time += inference_time

        # Save individual test result
        test_result = {
            "question_id": i,
            "question": question,
            "response": response,
            "inference_time_seconds": round(inference_time, 2),
            "response_length_chars": len(response)
        }
        results["test_questions"].append(test_result)

        print(f"Inference time: {inference_time:.2f} seconds")
        print(f"Response length: {len(response)} characters")

    except Exception as e:
        error_result = {
            "question_id": i,
            "question": question,
            "error": str(e),
            "inference_time_seconds": None,
            "response_length_chars": None
        }
        results["test_questions"].append(error_result)
        print(f"Error: {e}")

# Calculate summary statistics
if times:
    avg_time = total_time / len(times)
    min_time = min(times)
    max_time = max(times)

    results["summary"] = {
        "total_questions_tested": len(times),
        "successful_tests": len(times),
        "failed_tests": len(test_questions) - len(times),
        "average_inference_time_seconds": round(avg_time, 2),
        "fastest_inference_seconds": round(min_time, 2),
        "slowest_inference_seconds": round(max_time, 2),
        "total_time_seconds": round(total_time, 2)
    }

    print(f"\n" + "=" * 60)
    print(f"TIMING SUMMARY:")
    print(f"Total questions tested: {len(times)}")
    print(f"Average inference time: {avg_time:.2f} seconds")
    print(f"Fastest inference: {min_time:.2f} seconds")
    print(f"Slowest inference: {max_time:.2f} seconds")
    print(f"Total time for all tests: {total_time:.2f} seconds")

# Save results to file
results_file = "inference_test_results.json"
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nResults saved to: {results_file}")

In [ ]:
# Upload test results to Hugging Face
if config.save_to_hf and config.hf_username:
    try:
        from huggingface_hub import upload_file

        # Model name for HF
        model_name = f"{config.hf_username}/{config.model_name}"

        # Upload the test results file
        upload_file(
            path_or_fileobj="inference_test_results.json",
            path_in_repo="inference_test_results.json",
            repo_id=model_name,
            repo_type="model"
        )

        print(f"Test results uploaded to: https://huggingface.co/{model_name}")

    except Exception as e:
        print(f"Error uploading test results: {e}")
else:
    print("Skipping test results upload to Hugging Face")

In [ ]:
# Download the test results file
from google.colab import files
files.download("inference_test_results.json")